### Step 1: Comprehensive Environment Setup
This cell installs all the specific libraries needed to create a stable, reproducible JAX/Flax environment on Kaggle TPUs. We are pinning the exact versions for libraries like `jax`, `transformers`, and `flax` to prevent dependency conflicts, which is a common issue in complex machine learning projects. [cite_start]This single, comprehensive command ensures that our setup is reliable every time we run the notebook [cite: 43-53].

---

In [ ]:
# STEP 1: COMPREHENSIVE ENVIRONMENT SETUP
!pip install "numpy==1.26.4" \
"jax[tpu]==0.4.23" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html \
"transformers==4.43.3" \
"flax==0.7.5" \
"optax==0.2.2" \
"datasets" \
"sentencepiece" \
"orbax-checkpointing" \
"pyyaml" \
--quiet

### Step 2: Verify TPU Connection
This script serves as a crucial sanity check to confirm that our notebook is properly connected to the TPU accelerator. For the "TPU v3-8" environment, we expect to see exactly 8 available devices (one for each core). Verifying this now ensures that the data parallelism we will implement in Phase 3 using `jax.pmap` will work as intended by distributing the workload across all cores.

---

In [ ]:
import jax

# Check the number of available TPU devices
device_count = jax.device_count()
print(f"✅ Found {device_count} JAX devices (TPU cores).")

if device_count != 8:
    print("⚠️ Warning: Expected 8 TPU cores, but found a different number. Check your accelerator settings.")

### Step 3: Load Tokenizer and Model from Local Dataset
This is the key step that leverages the Kaggle dataset we created earlier. Instead of downloading the 16 GB model from the internet every time, we load it directly from the local `/kaggle/input/` path. This makes our notebook start up significantly faster and avoids potential network errors.

There are two important parameters in the model loading function:
* `dtype=jax.numpy.bfloat16`: We specify this data type because it is memory-efficient and highly optimized for performance on TPUs.
* `from_pt=True`: This flag tells the function to correctly convert the model's weights from their original PyTorch format into the JAX-native Flax format that we need for training.

**Note**: Remember to update the `local_model_path` variable to match the name of your specific Kaggle dataset.

---

In [3]:
import jax
from transformers import AutoTokenizer, FlaxAutoModelForCausalLM

# Point to the local path of your Kaggle dataset
# Make sure to update this path to match the name of your dataset!
local_model_path = "/kaggle/input/downloading-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Loading tokenizer from local path: {local_model_path}...")
tokenizer = AutoTokenizer.from_pretrained(local_model_path)
print("✅ Tokenizer loaded.")

print(f"Loading Flax model from local path: {local_model_path}...")
model = FlaxAutoModelForCausalLM.from_pretrained(
    local_model_path,
    dtype=jax.numpy.bfloat16, # Use bfloat16 for TPU memory efficiency
    from_pt=True              # Ensure correct conversion from PyTorch
)
print("✅ Flax model loaded successfully.")

Loading tokenizer from local path: /kaggle/input/downloading-llama/Meta-Llama-3.1-8B-Instruct...
✅ Tokenizer loaded.
Loading Flax model from local path: /kaggle/input/downloading-llama/Meta-Llama-3.1-8B-Instruct...


NotImplementedError: Support for sharded checkpoints using safetensors is coming soon!

### Step 4: Final Verification
This final, simple command prints the model's configuration to the console. Seeing the detailed output—including things like the number of layers, hidden size, and attention heads—is our definitive confirmation that the model object has been successfully and completely loaded into memory. This signals the successful completion of the entire environment preparation phase.

In [ ]:
# Print the model config to confirm successful loading
print("\n--- Model Configuration ---")
print(model.config)